# 03 — Building an Eval Harness

## Why this notebook exists

In **`02_assertions_and_golden_outputs.ipynb`** we built five grader functions — exact match, contains, regex, schema validation, golden outputs — each following the same shape: take an example and an agent output, return a score. That was a useful pattern, but we applied each grader by hand, one call at a time, against a single example. When you have a dataset of dozens or hundreds of examples and several graders to apply, doing that by hand doesn't scale.

This notebook generalizes the pattern into a **reusable eval harness**: a small set of dataclasses that model the inputs (`Example`), the per-example results (`ExampleResult`), and the aggregated report (`EvalReport`), plus a `run_eval(agent, dataset, graders)` function that wires them together. By the end you'll be able to add a new grader or a new dataset and re-run your entire eval suite in one call.

No API key needed — the agent is a deterministic Python stub and all graders run in-process.

## What you'll learn

- How to model an evaluation as three composable pieces: an `Example` (input + expected), an `ExampleResult` (example + output + scores), and an `EvalReport` (aggregated results with metrics).
- How to write `run_eval(agent, dataset, graders) -> EvalReport` — a runner that applies every grader to every example and collects structured results.
- How to read an `EvalReport`: `pass_rate()`, `mean_score()`, per-grader breakdowns, and `summary_table()`.
- Why pass rate and mean score tell different stories about the same run.
- A small but real Gotcha: graders from notebook 02 read `example["expected"]` (dict form). Now that `Example` is a dataclass, they must read `example.expected` (attribute form). The graders in this notebook are re-declared to match.

## 1. Setup

We re-declare `Score` and every deterministic grader from notebook 02 here, so this notebook is fully self-contained and runs in a fresh kernel without importing from another file.

**One small change from notebook 02:** the graders there read `example["expected"]` because examples were plain dicts. In this notebook, `Example` is a dataclass, so the graders read `example.expected` (attribute access). Everything else is identical — same function names, same signatures, same `Score` return type.

> **Gotcha:** If you copy a grader verbatim from notebook 02 and forget to change `example["expected"]` to `example.expected`, you'll get a `TypeError: 'Example' object is not subscriptable`. The fix is one character: replace `[` with `.` and drop the `"]"`. This is the only breaking change between the two notebooks.

In [ ]:
from __future__ import annotations

import re
import json
from dataclasses import dataclass, field
from typing import Any, Callable

from pydantic import BaseModel, ValidationError


# ---------------------------------------------------------------------------
# Score — the atomic unit every grader returns
# ---------------------------------------------------------------------------

@dataclass
class Score:
    key: str          # grader identity, e.g. "exact_match"
    score: float      # numeric value in [0, 1]
    passed: bool      # True if this grader considers the output acceptable
    comment: str = "" # optional human-readable note


# ---------------------------------------------------------------------------
# Deterministic graders — re-declared from notebook 02, now reading
# example.expected (attribute form) instead of example["expected"] (dict form)
# ---------------------------------------------------------------------------

def exact_match(example, output) -> Score:
    """Pass if str(output) == str(example.expected), case-insensitive strip."""
    expected = str(example.expected).strip().lower()
    actual = str(output).strip().lower()
    passed = expected == actual
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment=f"expected={expected!r}, got={actual!r}",
    )


def make_contains(substring: str, key: str = "contains") -> Callable:
    """Return a grader that passes when `substring` appears in the output."""
    def grader(example, output) -> Score:
        passed = substring.lower() in str(output).lower()
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment=f"looked for {substring!r}",
        )
    return grader


def make_regex(pattern: str, key: str = "regex") -> Callable:
    """Return a grader that passes when `pattern` matches anywhere in the output."""
    compiled = re.compile(pattern, re.IGNORECASE)
    def grader(example, output) -> Score:
        passed = bool(compiled.search(str(output)))
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment=f"pattern={pattern!r}",
        )
    return grader


def make_schema_grader(model: type[BaseModel], key: str = "valid_schema") -> Callable:
    """Return a grader that passes when the output parses into `model`."""
    def grader(example, output) -> Score:
        try:
            if isinstance(output, str):
                data = json.loads(output)
            else:
                data = output
            model.model_validate(data)
            return Score(key=key, score=1.0, passed=True, comment="schema valid")
        except (ValidationError, json.JSONDecodeError, TypeError) as exc:
            return Score(key=key, score=0.0, passed=False, comment=str(exc))
    return grader


def make_golden_grader(path: str, key: str = "golden") -> Callable:
    """Return a grader that passes when output matches the text stored at `path`."""
    import pathlib
    golden = pathlib.Path(path).read_text().strip()
    def grader(example, output) -> Score:
        actual = str(output).strip()
        passed = actual == golden
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment=f"golden={golden[:40]!r}",
        )
    return grader


print("Setup OK — Score and graders declared (attribute form)")